# app

> FastHTML viewer: auth gate + month index + month view + authed downloads

In [ ]:
#| default_exp app

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, secrets
from pathlib import Path
from fasthtml.common import fast_app, Beforeware, Titled, Form, Input, Button, P, Ul, Li, A, NotStr
from starlette.responses import RedirectResponse, FileResponse
from starlette.exceptions import HTTPException
from reconcile_web.archive import list_months, month_counts, status_html, safe_file

In [ ]:
#| export
def create_app(
    archive_dir: str|Path|None = None,  # archive root (default: env ARCHIVE_DIR)
    password: str|None = None,          # shared password (default: env APP_PASSWORD)
    session_secret: str|None = None,    # session signing key (default: env SESSION_SECRET)
):
    "Build the FastHTML viewer app; raise RuntimeError on missing/invalid config"
    archive_dir = archive_dir or os.environ.get('ARCHIVE_DIR')
    app_password = password or os.environ.get('APP_PASSWORD')
    session_secret = session_secret or os.environ.get('SESSION_SECRET')
    missing = [n for n, v in (('ARCHIVE_DIR', archive_dir), ('APP_PASSWORD', app_password),
                              ('SESSION_SECRET', session_secret)) if not v]
    if missing: raise RuntimeError(f"missing config: {', '.join(missing)}")
    if not Path(archive_dir).is_dir(): raise RuntimeError(f"ARCHIVE_DIR is not a directory: {archive_dir}")

    def auth_before(req, session):
        if not session.get('auth'): return RedirectResponse('/login', status_code=303)

    # skip only /login: the app serves no local static files (Pico CSS comes from the CDN), and
    # FastHTML's default static route can serve pdf/csv from cwd — an exempted path would bypass auth
    app, rt = fast_app(before=Beforeware(auth_before, skip=[r'/login']),
                       secret_key=session_secret, sess_https_only=True)

    def login_page(error=False):
        body = [Form(Input(name='password', type='password', required=True),
                     Button('Login'), method='post', action='/login')]
        if error: body.insert(0, P('Wrong password'))
        return Titled('Login', *body)

    @rt('/login', methods=['GET'])
    def login_form(): return login_page()

    @rt('/login', methods=['POST'])
    def login_submit(password: str, session):
        if password and secrets.compare_digest(password, app_password):
            session['auth'] = True
            return RedirectResponse('/', status_code=303)
        return login_page(error=True)

    @rt('/logout', methods=['GET'])
    def logout(session):
        session.pop('auth', None)
        return RedirectResponse('/login', status_code=303)

    @rt('/', methods=['GET'])
    def index():
        items = [Li(A(m, href=f'/m/{m}'),
                    f" — collected {c.get('collected', 0)}, not-needed {c.get('not-needed', 0)},"
                    f" missing {c.get('missing', 0)}")
                 for m in list_months(archive_dir) for c in [month_counts(archive_dir, m)]]
        return Titled('reconcile-archive', Ul(*items), P(A('Logout', href='/logout')))

    return app

In [ ]:
# test fixture: a throwaway archive matching the status.md contract in the spec
import tempfile
from pathlib import Path

STATUS_MD = """# {m} — receipt status

| date | title | amount | status | receipt_file | note |
|---|---|---|---|---|---|
| {m}-29 | OPENAI *CHATGPT SUBSCR | -110.46 | collected | [{m}-29_openai.pdf](receipts/{m}-29_openai.pdf) | src=x.pdf |
| {m}-06 | NORDEA | -20.46 | not-needed |  |  |

**missing (retrievable from Gmail/portal/Nordea): 1**
"""

def make_archive(months=('2025-07', '2025-08')):
    root = Path(tempfile.mkdtemp())
    (root/'README.md').write_text('readme')
    (root/'notamonth').mkdir()
    for m in months:
        d = root/m
        (d/'receipts').mkdir(parents=True)
        (d/'statement.pdf').write_bytes(b'%PDF-1.4 fake')
        (d/'statement.csv').write_text('date;amount\n')
        (d/'status.md').write_text(STATUS_MD.format(m=m))
        (d/'receipts'/f'{m}-29_openai.pdf').write_bytes(b'%PDF-1.4 fake receipt')
    return root

In [ ]:
from starlette.testclient import TestClient

root = make_archive()
app = create_app(archive_dir=root, password='pw', session_secret='s3cret')
client = TestClient(app, base_url='https://testserver')  # https: Secure cookie must round-trip

# unauthenticated -> redirected to /login
r = client.get('/', follow_redirects=False)
assert r.status_code == 303 and r.headers['location'] == '/login'
# static-looking paths are NOT auth-exempt (FastHTML's static route can serve pdf/csv)
assert client.get('/static/x.pdf', follow_redirects=False).status_code == 303
# login page reachable without auth
assert client.get('/login').status_code == 200
# wrong password -> error notice, still no access
r = client.post('/login', data={'password': 'wrong'})
assert 'Wrong password' in r.text
assert client.get('/', follow_redirects=False).status_code == 303
# correct password -> index with months and counts
r = client.post('/login', data={'password': 'pw'}, follow_redirects=False)
assert r.status_code == 303
r = client.get('/')
assert r.status_code == 200 and '2025-07' in r.text and '2025-08' in r.text
assert 'collected 1' in r.text and 'not-needed 1' in r.text and 'missing 1' in r.text
# logout ends the session
client.get('/logout', follow_redirects=False)
assert client.get('/', follow_redirects=False).status_code == 303
# missing config fails fast
try:
    create_app(archive_dir=root, password='pw'); assert False, 'should have raised'
except RuntimeError as e:
    assert 'SESSION_SECRET' in str(e)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()